# ColBERTv2: Indexing & Search Notebook

We start by importing the relevant classes. As we'll see below, `Indexer` and `Searcher` are the key actors here. 

In [ ]:
!pip3 install ujson huggingface_hub


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import sys
sys.path.insert(0, '../')

from colbert.infra import Run, RunConfig, ColBERTConfig
from colbert.data import Queries, Collection
from colbert import Indexer, Searcher

The workflow here assumes an IR dataset: a set of queries and a corresponding collection of passages.

The classes `Queries` and `Collection` provide a convenient interface for working with such datasets.

We will use the *dev set* of the **LoTTE benchmark** we recently introduced in the ColBERTv2 paper. The dev and test sets contain several domain-specific corpora, and we'll use the smallest dev set corpus, namely `lifestyle:dev`.

In [3]:
!mkdir -p downloads/

# ColBERTv2 checkpoint trained on MS MARCO Passage Ranking (388MB compressed)
!wget https://downloads.cs.stanford.edu/nlp/data/colbert/colbertv2/colbertv2.0.tar.gz -P downloads/
!tar -xvzf downloads/colbertv2.0.tar.gz -C downloads/

# The LoTTE dev and test sets (3.4GB compressed)
!wget https://downloads.cs.stanford.edu/nlp/data/colbert/colbertv2/lotte.tar.gz -P downloads/
!tar -xvzf downloads/lotte.tar.gz -C downloads/

--2025-02-27 15:55:06--  https://downloads.cs.stanford.edu/nlp/data/colbert/colbertv2/colbertv2.0.tar.gz
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 405924985 (387M) [application/octet-stream]
Saving to: ‘downloads/colbertv2.0.tar.gz’

colbertv2.0.tar.gz  100%[===================>] 387.12M  1.93MB/s    in 4m 31s  

2025-02-27 15:59:39 (1.43 MB/s) - ‘downloads/colbertv2.0.tar.gz’ saved [405924985/405924985]

x colbertv2.0/
x colbertv2.0/artifact.metadata
x colbertv2.0/vocab.txt
x colbertv2.0/tokenizer.json
x colbertv2.0/special_tokens_map.json
x colbertv2.0/tokenizer_config.json
x colbertv2.0/config.json
x colbertv2.0/pytorch_model.bin
--2025-02-27 15:59:41--  https://downloads.cs.stanford.edu/nlp/data/colbert/colbertv2/lotte.tar.gz
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 

In [6]:
dataroot = 'downloads/lotte'
dataset = 'lifestyle'
datasplit = 'dev'

queries = os.path.join(dataroot, dataset, datasplit, 'questions.search.tsv')
collection = os.path.join(dataroot, dataset, datasplit, 'collection.tsv')

queries = Queries(path=queries)
collection = Collection(path=collection)

f'Loaded {len(queries)} queries and {len(collection):,} passages'

[Feb 27, 18:14:16] #> Loading the queries from downloads/lotte/lifestyle/dev/questions.search.tsv ...
[Feb 27, 18:14:16] #> Got 417 queries. All QIDs are unique.

[Feb 27, 18:14:16] #> Loading collection...
0M 


'Loaded 417 queries and 268,893 passages'

This loaded 417 queries and 269k passages. Let's inspect one query and one passage.

In [8]:
os.path.join(dataroot, dataset, datasplit, 'collection.tsv')

'downloads/lotte/lifestyle/dev/collection.tsv'

In [10]:
os.path.join(dataroot, dataset, datasplit, 'questions.search.tsv')

'downloads/lotte/lifestyle/dev/questions.search.tsv'

In [9]:
!pwd

/Users/pragalbh.devsingh/Documents/work/models/ColBERT/docs


In [1]:
50000000/1024**2

47.6837158203125

In [5]:
print(queries[24])
print()
print(collection[89852])
print()

are blossom end rot tomatoes edible?

I don't know what J-Rock is definitively as it could mean a number of things, however if you mean a 'metal' style rock guitar sound that I believe to be popular over there, then it can be quite simple: Strip all of your pedals and effects, and turn all dials to zero; Turn the volume up as far as it will go; Keep the Bass and Treble dials high; Turn down any 'Middle' dials until the sound becomes scooped - bear in the mind that middle frequencies are important for clarity, so find a decent compromise; Bring up the gain until you get the sound you require; Finally, bring the volume down to a more manageable level. This will get you a lot of the way, but without any clearer description of what you want to sound like, then I can't offer any more help.



## Indexing

For efficient search, we can pre-compute the ColBERT representation of each passage and index them.

Below, the `Indexer` take a model checkpoint and writes a (compressed) index to disk. We then prepare a `Searcher` for retrieval from this index.

(With four Titan V GPUs, indexing should take about 13 minutes. The output is fairly long/ugly at the moment!)

In [ ]:
nbits = 2   # encode each dimension with 2 bits
doc_maxlen = 300   # truncate passages at 300 tokens

checkpoint = 'downloads/colbertv2.0'
index_name = f'{dataset}.{datasplit}.{nbits}bits'

In [ ]:
with Run().context(RunConfig(nranks=4, experiment='notebook')):  # nranks specifies the number of GPUs to use.
    config = ColBERTConfig(doc_maxlen=doc_maxlen, nbits=nbits)

    indexer = Indexer(checkpoint=checkpoint, config=config)
    indexer.index(name=index_name, collection=collection, overwrite=True)

In [ ]:
indexer.get_index() # You can get the absolute path of the index, if needed.

## Search

Having built the index and prepared our `searcher`, we can search for individual query strings.

We can use the `queries` set we loaded earlier — or you can supply your own questions. Feel free to get creative! But keep in mind this set of ~300k lifestyle passages can only answer a small, focused set of questions!

In [ ]:
# To create the searcher using its relative name (i.e., not a full path), set
# experiment=value_used_for_indexing in the RunConfig.
with Run().context(RunConfig(experiment='notebook')):
    searcher = Searcher(index=index_name)


# If you want to customize the search latency--quality tradeoff, you can also supply a
# config=ColBERTConfig(ncells=.., centroid_score_threshold=.., ndocs=..) argument.
# The default settings with k <= 10 (1, 0.5, 256) gives the fastest search,
# but you can gain more extensive search by setting larger values of k or
# manually specifying more conservative ColBERTConfig settings (e.g. (4, 0.4, 4096)).

In [ ]:
query = queries[37]   # or supply your own query

print(f"#> {query}")

# Find the top-3 passages for this query
results = searcher.search(query, k=3)

# Print out the top-k retrieved passages
for passage_id, passage_rank, passage_score in zip(*results):
    print(f"\t [{passage_rank}] \t\t {passage_score:.1f} \t\t {searcher.collection[passage_id]}")

## Batch Search

In many applications, you have a large batch of queries and you need to maximize the overall throughput. For that, you can use the `searcher.search_all(queries, k)` method, which returns a `Ranking` object that organizes the results across all queries.

(Batching provides many opportunities for higher-throughput search, though we have not implemented most of those optimizations for compressed indexes yet.)

In [ ]:
rankings = searcher.search_all(queries, k=5).todict()

In [ ]:
rankings[30]  # For query 30, a list of (passage_id, rank, score) for the top-k passages